In [1]:
%pip install numpy pandas scikit-learn xgboost ucimlrepo matplotlib seaborn joblib ipykernel jupyter


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV
)

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    classification_report,
    ConfusionMatrixDisplay
)

# ==========================================
# 2. Project Directory
# ==========================================

# Locate the project root
ROOT = Path.cwd() if (Path.cwd() / "src" / "data.py").exists() else Path.cwd().parent

# Add the project root to Python path
sys.path.insert(0, str(ROOT))

# Import shared functions
from src.data import load_dataset, split, make_preprocessor


# ==========================================
# 3. Experiment Settings
# ==========================================

dataset_names = ["adult", "ai4i"]

split_seed = 42
validation_split = 0.25

experiment_seeds = [42, 43, 44]


# ==========================================
# 4. Output Directories
# ==========================================

results_dir = ROOT / "results" / "logistic_regression"
models_dir = ROOT / "models" / "logistic_regression"

results_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)


# ==========================================
# 5. Confirm Setup
# ==========================================

print("Setup completed successfully!")
print("Project root:", ROOT)
print("Datasets:", dataset_names)

Setup completed successfully!
Project root: /Users/zhanghaoran/Documents/zyw/UC_davis/Subjects/ECS171/ECS-171-Project
Datasets: ['adult', 'ai4i']


In [3]:
# ==========================================
# Load Data and Create Splits
# ==========================================

clean_data = {}
split_sizes = []

for dataset_name in dataset_names:

    print(f"Loading dataset: {dataset_name}")

    # Load dataset
    X, y = load_dataset(dataset_name)

    # 80% training + validation, 20% testing
    X_train_full, X_test, y_train_full, y_test = split(
        X, y, seed=split_seed
    )

    # Split the remaining 80% into 60% train and 20% validation
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full,
        y_train_full,
        test_size=validation_split,
        stratify=y_train_full,
        random_state=split_seed
    )

    # Store datasets
    clean_data[dataset_name] = {
        "X_train": X_train,
        "X_val": X_val,
        "X_test": X_test,
        "y_train": y_train,
        "y_val": y_val,
        "y_test": y_test
    }

    # Record dataset sizes
    split_sizes.append({
        "dataset": dataset_name,
        "train_size": len(X_train),
        "val_size": len(X_val),
        "test_size": len(X_test),
        "positive_rate": y.mean()
    })

# Display summary
split_df = pd.DataFrame(split_sizes)
display(split_df)

Loading dataset: adult
Loading dataset: ai4i


,dataset,train_size,val_size,test_size,positive_rate
0,adult,24863,8288,8288,0.239388
1,ai4i,6000,2000,2000,0.033900


In [4]:
# ==========================================
# Logistic Regression Pipeline
# ==========================================

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipelines = {}

for dataset_name in dataset_names:
    data = clean_data[dataset_name]

    pipelines[dataset_name] = Pipeline([
        ("preprocess", make_preprocessor(
            data["X_train"], mode="impute"
        )),
        ("model", LogisticRegression(
            solver="lbfgs",
            max_iter=3000,
            random_state=split_seed
        ))
    ])

    print(f"Pipeline created for {dataset_name}")

Pipeline created for adult
Pipeline created for ai4i


In [5]:
# ==========================================
# Tune Hyperparameters Using GridSearchCV
# ==========================================

# Define hyperparameter grid
param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__class_weight": [None, "balanced"]
}

# 5-fold stratified cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=split_seed
)

best_params = {}
searches = {}
search_results = []

# ==========================================
# Hyperparameter Tuning
# ==========================================

for dataset_name in dataset_names:

    print(f"\nTuning Logistic Regression on {dataset_name}")

    data = clean_data[dataset_name]

    search = GridSearchCV(
        estimator=pipelines[dataset_name],
        param_grid=param_grid,
        scoring="average_precision",
        cv=cv,
        n_jobs=-1,
        refit=False,
        verbose=1
    )

    # Fit GridSearchCV
    search.fit(
        data["X_train"],
        data["y_train"]
    )

    # Store search object
    searches[dataset_name] = search

    # Extract best hyperparameters
    best_params[dataset_name] = {
        key.replace("model__", ""): value
        for key, value in search.best_params_.items()
    }

    # Store results
    search_results.append({
        "dataset": dataset_name,
        "best_C": best_params[dataset_name]["C"],
        "best_class_weight": str(
            best_params[dataset_name]["class_weight"]
        ),
        "best_cv_average_precision": search.best_score_
    })


# ==========================================
# Display Best Hyperparameters as DataFrame
# ==========================================

best_params_df = pd.DataFrame(search_results)

print("\n===== Best Hyperparameters =====")
print(best_params_df.round(4).to_string(index=False))


Tuning Logistic Regression on adult
Fitting 5 folds for each of 10 candidates, totalling 50 fits

Tuning Logistic Regression on ai4i
Fitting 5 folds for each of 10 candidates, totalling 50 fits

===== Best Hyperparameters =====
dataset  best_C best_class_weight  best_cv_average_precision
  adult       1              None                     0.7698
   ai4i      10              None                     0.4722


In [6]:
# ==========================================
# Train Logistic Regression
# ==========================================

normal_models = {}
training_times = {}

for dataset_name in dataset_names:

    print(f"\nTraining Logistic Regression on {dataset_name}")

    data = clean_data[dataset_name]

    for seed in experiment_seeds:

        # Define pipeline with best hyperparameters
        model = Pipeline([
            ("preprocess", make_preprocessor(
                data["X_train"], mode="impute"
            )),
            ("model", LogisticRegression(
                solver="lbfgs",
                max_iter=3000,
                random_state=seed,
                **best_params[dataset_name]
            ))
        ])

        # Measure training time
        start_time = time.perf_counter()

        model.fit(
            data["X_train"],
            data["y_train"]
        )

        training_times[(dataset_name, seed)] = (
            time.perf_counter() - start_time
        )

        # Store trained model
        normal_models[(dataset_name, seed)] = model

        print(
            f"Training time for {dataset_name}, "
            f"seed {seed}: "
            f"{training_times[(dataset_name, seed)]:.2f} seconds"
        )



Training Logistic Regression on adult
Training time for adult, seed 42: 0.19 seconds
Training time for adult, seed 43: 0.15 seconds
Training time for adult, seed 44: 0.15 seconds

Training Logistic Regression on ai4i
Training time for ai4i, seed 42: 0.01 seconds
Training time for ai4i, seed 43: 0.01 seconds
Training time for ai4i, seed 44: 0.02 seconds


In [7]:
# ==========================================
# Validation
# ==========================================

normal_val_metrics = []

for dataset_name in dataset_names:

    data = clean_data[dataset_name]

    X_val = data["X_val"]
    y_val = data["y_val"]

    for seed in experiment_seeds:

        model = normal_models[(dataset_name, seed)]

        # Predictions
        y_pred = model.predict(X_val)
        y_score = model.predict_proba(X_val)[:, 1]

        # Metrics
        normal_val_metrics.append({
            "dataset": dataset_name,
            "seed": seed,
            "accuracy": accuracy_score(y_val, y_pred),
            "precision": precision_score(
                y_val, y_pred, zero_division=0
            ),
            "recall": recall_score(
                y_val, y_pred, zero_division=0
            ),
            "f1": f1_score(
                y_val, y_pred, zero_division=0
            ),
            "average_precision": average_precision_score(
                y_val, y_score
            ),
            "training_seconds": training_times[
                (dataset_name, seed)
            ]
        })

normal_val_df = pd.DataFrame(normal_val_metrics)

print("\nValidation Results:")
print(normal_val_df.round(4).to_string(index=False))


Validation Results:
dataset  seed  accuracy  precision  recall     f1  average_precision  training_seconds
  adult    42    0.8568     0.7455  0.6099 0.6709             0.7815            0.1865
  adult    43    0.8568     0.7455  0.6099 0.6709             0.7815            0.1526
  adult    44    0.8568     0.7455  0.6099 0.6709             0.7815            0.1508
   ai4i    42    0.9695     0.6842  0.1912 0.2989             0.4045            0.0143
   ai4i    43    0.9695     0.6842  0.1912 0.2989             0.4045            0.0136
   ai4i    44    0.9695     0.6842  0.1912 0.2989             0.4045            0.0157


In [8]:
# ==========================================
# Test
# ==========================================

normal_test_metrics = []

for dataset_name in dataset_names:

    data = clean_data[dataset_name]

    X_test = data["X_test"]
    y_test = data["y_test"]

    for seed in experiment_seeds:

        model = normal_models[(dataset_name, seed)]

        # Predictions
        y_pred = model.predict(X_test)
        y_score = model.predict_proba(X_test)[:, 1]

        # Metrics
        normal_test_metrics.append({
            "dataset": dataset_name,
            "seed": seed,
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(
                y_test, y_pred, zero_division=0
            ),
            "recall": recall_score(
                y_test, y_pred, zero_division=0
            ),
            "f1": f1_score(
                y_test, y_pred, zero_division=0
            ),
            "average_precision": average_precision_score(
                y_test, y_score
            )
        })

normal_test_df = pd.DataFrame(normal_test_metrics)

print("\nTest Results:")
print(normal_test_df.round(4).to_string(index=False))


Test Results:
dataset  seed  accuracy  precision  recall     f1  average_precision
  adult    42     0.859     0.7495  0.6169 0.6768             0.7819
  adult    43     0.859     0.7495  0.6169 0.6768             0.7819
  adult    44     0.859     0.7495  0.6169 0.6768             0.7819
   ai4i    42     0.968     0.6250  0.1471 0.2381             0.4171
   ai4i    43     0.968     0.6250  0.1471 0.2381             0.4171
   ai4i    44     0.968     0.6250  0.1471 0.2381             0.4171


In [9]:
# ==========================================
# Compare Validation and Test Performance
# ==========================================

metrics = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "average_precision"
]

val_summary = (
    normal_val_df
    .groupby("dataset")[metrics]
    .mean()
)

test_summary = (
    normal_test_df
    .groupby("dataset")[metrics]
    .mean()
)

comparison_df = pd.concat(
    {
        "validation": val_summary,
        "test": test_summary
    },
    axis=1
)

print("\nComparison Results:")
print(comparison_df.round(4).to_string(index=False))


Comparison Results:
validation                                               test                                          
  accuracy precision recall     f1 average_precision accuracy precision recall     f1 average_precision
    0.8568    0.7455 0.6099 0.6709            0.7815    0.859    0.7495 0.6169 0.6768            0.7819
    0.9695    0.6842 0.1912 0.2989            0.4045    0.968    0.6250 0.1471 0.2381            0.4171


Logistic Regression performs reasonably well on Adult Income, but struggles to identify actual machine failures in AI4I, despite achieving 96.8% accuracy.

In [10]:
# ==========================================
# Save Models and Results
# ==========================================

# Save validation and test metrics
normal_val_df.to_csv(
    results_dir / "validation_metrics.csv",
    index=False
)

normal_test_df.to_csv(
    results_dir / "test_metrics.csv",
    index=False
)

# Save one trained model for each dataset
for dataset_name in dataset_names:

    model = normal_models[(dataset_name, 42)]

    model_path = models_dir / f"{dataset_name}_clean.joblib"

    joblib.dump(model, model_path)

    print(f"Saved model: {model_path.name}")




Saved model: adult_clean.joblib
Saved model: ai4i_clean.joblib

Models and results saved successfully!
